# Notebook 11 — LSTM + Bahdanau Attention

Four experiments: Vanilla LSTM and LSTM+Bahdanau Attention, each on the full
feature set and the price-only subset. The attention model is the project's
Novelty 2. The final cells consolidate all 8 experiments (including the tree
baselines from Notebook 10) into the 2×4 ablation table.

**Pipeline position.**

`features_targets.parquet` (Step 4) + `eval_results_baselines.json` (Nb10) → **this notebook** → `eval_results_lstm.json` + ablation table.


## Setup

In [1]:
import os, json, warnings, itertools
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report
)

warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

Device: cuda
PyTorch: 2.10.0+cu128


In [2]:
# --- CONFIG ---
INPUT_DIR  = "/kaggle/input"
OUTPUT_DIR = "/kaggle/working"
FIG_DIR    = f"{OUTPUT_DIR}/figures"
os.makedirs(FIG_DIR, exist_ok=True)

DATA_PATH         = f"{INPUT_DIR}/datasets/leev75/pfe-step41/features_targets.parquet"
BASELINE_RESULTS  = f"{OUTPUT_DIR}/eval_results_baselines.json"
RESULTS_PATH      = f"{OUTPUT_DIR}/eval_results_lstm.json"

TARGET_COL  = "target_X1.0"
LABEL_MAP   = {"BUY": 0, "HOLD": 1, "SELL": 2}
LABEL_NAMES = ["BUY", "HOLD", "SELL"]
N_CLASSES   = 3

META_COLS = ["trading_day", "split", "ret_continuous",
             "target_X0.5", "target_X1.0", "target_X1.5", "target_X2.0"]
PRICE_BASE = ["open", "high", "low", "close", "adj_close", "volume_shares"]

# Hyperparameter search space.
HP_SEQ_LENGTHS  = [5, 10, 20]
HP_HIDDEN_SIZES = [64, 128]
HP_NUM_LAYERS   = [1, 2]
HP_DROPOUTS     = [0.2, 0.3, 0.5]

# Training constants.
MAX_EPOCHS     = 100
BATCH_SIZE     = 32
LR             = 1e-3
PATIENCE       = 10
LR_PATIENCE    = 5
LR_FACTOR      = 0.5
GRAD_CLIP      = 1.0

SEED        = 42
N_BOOTSTRAP = 1000

torch.manual_seed(SEED)
np.random.seed(SEED)
print("Config loaded.")

Config loaded.


## 1. Load data and prepare feature sets

In [3]:
df = pd.read_parquet(DATA_PATH)
train = df[df["split"] == "train"].copy()
test  = df[df["split"] == "test"].copy()

all_features = [c for c in df.columns if c not in META_COLS]
price_features = [c for c in all_features
                  if any(c == p or c.startswith(p + "_") for p in PRICE_BASE)]

for d in [train, test]:
    d["y"] = d[TARGET_COL].map(LABEL_MAP)

y_train_arr = train["y"].values
y_test_arr  = test["y"].values
ret_test    = test["ret_continuous"].values

# Compute class weights.
from collections import Counter
counts = Counter(y_train_arr)
total  = len(y_train_arr)
class_weights = torch.tensor(
    [total / (N_CLASSES * counts[i]) for i in range(N_CLASSES)],
    dtype=torch.float32
).to(device)
print(f"Class weights: {class_weights.cpu().tolist()}")
print(f"Train: {len(train)}, Test: {len(test)}")
print(f"Full features: {len(all_features)}, Price-only: {len(price_features)}")

Class weights: [0.8073592782020569, 1.883838415145874, 0.812636137008667]
Train: 746, Test: 249
Full features: 230, Price-only: 5


## 2. Windowing and scaling helpers

In [4]:
def scale_and_window(train_df, test_df, feature_cols, seq_len):
    """Scale features with StandardScaler (fit on train), then create
    sliding-window 3D arrays for LSTM input.

    Returns: X_train (N_tr, L, F), y_train (N_tr,),
             X_test (N_te, L, F), y_test (N_te,), ret_test (N_te,)
    """
    scaler = StandardScaler()
    X_tr_flat = scaler.fit_transform(train_df[feature_cols].values)
    X_te_flat = scaler.transform(test_df[feature_cols].values)
    y_tr = train_df["y"].values
    y_te = test_df["y"].values
    r_te = test_df["ret_continuous"].values

    def _window(X, y, r=None):
        Xw, yw, rw = [], [], []
        for i in range(len(X) - seq_len + 1):
            Xw.append(X[i : i + seq_len])
            yw.append(y[i + seq_len - 1])
            if r is not None:
                rw.append(r[i + seq_len - 1])
        return np.array(Xw), np.array(yw), np.array(rw) if r is not None else None

    X_tr_w, y_tr_w, _   = _window(X_tr_flat, y_tr)
    X_te_w, y_te_w, r_w = _window(X_te_flat, y_te, r_te)
    return X_tr_w, y_tr_w, X_te_w, y_te_w, r_w


print("Windowing helper defined.")

Windowing helper defined.


## 3. Model definitions

In [5]:
class BahdanauAttention(nn.Module):
    """Single-head additive (Bahdanau) attention."""
    def __init__(self, hidden_size):
        super().__init__()
        self.W_h = nn.Linear(hidden_size, hidden_size, bias=True)
        self.v   = nn.Linear(hidden_size, 1, bias=False)

    def forward(self, lstm_outputs):
        # lstm_outputs: (batch, L, H)
        energy = self.v(torch.tanh(self.W_h(lstm_outputs)))  # (batch, L, 1)
        alpha  = F.softmax(energy.squeeze(-1), dim=1)        # (batch, L)
        context = (alpha.unsqueeze(2) * lstm_outputs).sum(1)  # (batch, H)
        return context, alpha


class LSTMClassifier(nn.Module):
    """LSTM with optional Bahdanau attention for 3-class classification."""
    def __init__(self, input_size, hidden_size, num_layers, dropout, use_attention=False):
        super().__init__()
        self.use_attention = use_attention
        self.lstm = nn.LSTM(
            input_size=input_size, hidden_size=hidden_size,
            num_layers=num_layers, dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )
        if use_attention:
            self.attention = BahdanauAttention(hidden_size)
        self.drop = nn.Dropout(dropout)
        self.fc   = nn.Linear(hidden_size, N_CLASSES)

    def forward(self, x):
        # x: (batch, L, F)
        lstm_out, _ = self.lstm(x)  # (batch, L, H)
        if self.use_attention:
            context, alpha = self.attention(lstm_out)
        else:
            context = lstm_out[:, -1, :]  # last timestep
            alpha = None
        out = self.fc(self.drop(context))  # (batch, 3)
        return out, alpha


print("Models defined: LSTMClassifier + BahdanauAttention")

Models defined: LSTMClassifier + BahdanauAttention


## 4. Training loop

In [6]:
def train_model(model, X_train, y_train, X_val, y_val,
                max_epochs=MAX_EPOCHS, patience=PATIENCE):
    """Train with early stopping on validation loss."""
    train_ds = TensorDataset(
        torch.tensor(X_train, dtype=torch.float32),
        torch.tensor(y_train, dtype=torch.long),
    )
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=False)

    X_val_t = torch.tensor(X_val, dtype=torch.float32).to(device)
    y_val_t = torch.tensor(y_val, dtype=torch.long).to(device)

    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=LR_FACTOR, patience=LR_PATIENCE, min_lr=1e-6
    )

    best_val_loss = float("inf")
    best_state    = None
    wait          = 0

    for epoch in range(max_epochs):
        model.train()
        for xb, yb in train_dl:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out, _ = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()

        # Validation.
        model.eval()
        with torch.no_grad():
            val_out, _ = model(X_val_t)
            val_loss = criterion(val_out, y_val_t).item()
        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break

    model.load_state_dict(best_state)
    return model, best_val_loss


def predict_model(model, X):
    """Run inference, return predicted class indices and attention weights."""
    model.eval()
    X_t = torch.tensor(X, dtype=torch.float32).to(device)
    with torch.no_grad():
        out, alpha = model(X_t)
        preds = out.argmax(dim=1).cpu().numpy()
        attn  = alpha.cpu().numpy() if alpha is not None else None
    return preds, attn


print("Training loop defined.")

Training loop defined.


## 5. Evaluation helpers

In [7]:
def compute_sharpe(y_pred_labels, ret_continuous):
    position_map = {"BUY": 1.0, "SELL": -1.0, "HOLD": 0.0}
    positions = np.array([position_map[l] for l in y_pred_labels])
    daily_ret = positions * ret_continuous
    if daily_ret.std() == 0:
        return 0.0
    return float(daily_ret.mean() / daily_ret.std() * np.sqrt(252))


def evaluate_experiment(y_true_int, y_pred_int, y_pred_labels, ret_continuous):
    da     = accuracy_score(y_true_int, y_pred_int)
    f1     = f1_score(y_true_int, y_pred_int, average="macro")
    sharpe = compute_sharpe(y_pred_labels, ret_continuous)
    prec = precision_score(y_true_int, y_pred_int, average=None, labels=[0,1,2], zero_division=0)
    rec  = recall_score(y_true_int, y_pred_int, average=None, labels=[0,1,2], zero_division=0)
    return {
        "DA": da, "macro_F1": f1, "Sharpe": sharpe,
        "precision_BUY": prec[0], "precision_HOLD": prec[1], "precision_SELL": prec[2],
        "recall_BUY": rec[0], "recall_HOLD": rec[1], "recall_SELL": rec[2],
    }


def bootstrap_ci(y_true_int, y_pred_int, y_pred_labels, ret_continuous, n_boot=N_BOOTSTRAP):
    n = len(y_true_int)
    boot = []
    for _ in range(n_boot):
        idx = np.random.randint(0, n, size=n)
        boot.append(evaluate_experiment(
            y_true_int[idx], y_pred_int[idx],
            [y_pred_labels[i] for i in idx], ret_continuous[idx]))
    boot_df = pd.DataFrame(boot)
    point = evaluate_experiment(y_true_int, y_pred_int, y_pred_labels, ret_continuous)
    return {m: {"point": point[m],
                "ci_low": float(np.percentile(boot_df[m], 2.5)),
                "ci_high": float(np.percentile(boot_df[m], 97.5))}
            for m in boot_df.columns}


def mcnemar_test(y_true, y_pred_a, y_pred_b):
    from scipy.stats import chi2
    ca, cb = (y_pred_a == y_true), (y_pred_b == y_true)
    b_not_a = (~ca & cb).sum()
    a_not_b = (ca & ~cb).sum()
    if b_not_a + a_not_b == 0:
        return {"statistic": 0.0, "p_value": 1.0}
    stat = (abs(b_not_a - a_not_b) - 1) ** 2 / (b_not_a + a_not_b)
    return {"statistic": float(stat), "p_value": float(1 - chi2.cdf(stat, df=1))}


print("Evaluation helpers defined.")

Evaluation helpers defined.


## 6. Hyperparameter search (walk-forward CV)

Grid search over {L, H, layers, dropout}. Uses the last CV fold's validation
loss to select the best combo. The search runs once on the full feature set;
the best architecture is reused for price-only (same model shape, different
input size).

In [8]:
def hp_search(train_df, feature_cols):
    """Grid search over LSTM hyperparameters using walk-forward CV."""
    tscv = TimeSeriesSplit(n_splits=5)
    F_dim = len(feature_cols)

    best_score = -1
    best_hp    = None
    results_log = []

    combos = list(itertools.product(
        HP_SEQ_LENGTHS, HP_HIDDEN_SIZES, HP_NUM_LAYERS, HP_DROPOUTS
    ))
    print(f"Searching {len(combos)} combinations...")

    for i, (L, H, N_lay, D) in enumerate(combos):
        fold_scores = []
        for fold_i, (tr_idx, val_idx) in enumerate(tscv.split(train_df)):
            tr_fold = train_df.iloc[tr_idx]
            va_fold = train_df.iloc[val_idx]

            X_tr, y_tr, X_va, y_va, _ = scale_and_window(
                tr_fold, va_fold, feature_cols, L)

            if len(X_tr) < BATCH_SIZE or len(X_va) < 2:
                continue

            model = LSTMClassifier(F_dim, H, N_lay, D, use_attention=True).to(device)
            model, _ = train_model(model, X_tr, y_tr, X_va, y_va,
                                   max_epochs=50, patience=5)

            preds, _ = predict_model(model, X_va)
            f1 = f1_score(y_va, preds, average="macro")
            fold_scores.append(f1)

        if fold_scores:
            mean_f1 = np.mean(fold_scores)
            results_log.append({"L": L, "H": H, "layers": N_lay, "D": D, "cv_f1": mean_f1})
            if mean_f1 > best_score:
                best_score = mean_f1
                best_hp = {"L": L, "H": H, "layers": N_lay, "D": D}

        if (i + 1) % 6 == 0:
            print(f"  {i+1}/{len(combos)} done, best so far: {best_hp} (F1={best_score:.4f})")

    print(f"\nBest HP: {best_hp}, CV macro-F1: {best_score:.4f}")
    return best_hp, pd.DataFrame(results_log)


best_hp, hp_log = hp_search(train, all_features)
print()
print("Top 5 configurations:")
print(hp_log.nlargest(5, "cv_f1").to_string(index=False))

Searching 36 combinations...
  6/36 done, best so far: {'L': 5, 'H': 64, 'layers': 1, 'D': 0.5} (F1=0.2979)
  12/36 done, best so far: {'L': 5, 'H': 128, 'layers': 1, 'D': 0.5} (F1=0.3123)
  18/36 done, best so far: {'L': 5, 'H': 128, 'layers': 1, 'D': 0.5} (F1=0.3123)
  24/36 done, best so far: {'L': 10, 'H': 128, 'layers': 2, 'D': 0.5} (F1=0.3294)
  30/36 done, best so far: {'L': 10, 'H': 128, 'layers': 2, 'D': 0.5} (F1=0.3294)
  36/36 done, best so far: {'L': 10, 'H': 128, 'layers': 2, 'D': 0.5} (F1=0.3294)

Best HP: {'L': 10, 'H': 128, 'layers': 2, 'D': 0.5}, CV macro-F1: 0.3294

Top 5 configurations:
 L   H  layers   D    cv_f1
10 128       2 0.5 0.329382
20 128       1 0.3 0.320547
 5 128       1 0.5 0.312320
20 128       2 0.3 0.300471
 5  64       1 0.5 0.297882


## 7. Train final models and evaluate

In [9]:
all_results = {}
predictions = {}  # store for McNemar later

for feat_label, feat_cols in [("full", all_features), ("price", price_features)]:
    for attn_label, use_attn in [("LSTM", False), ("Attn", True)]:
        name = f"{attn_label}_{feat_label}"
        print(f"\n{'='*60}")
        print(f"Experiment: {name}")
        print(f"{'='*60}")

        L = best_hp["L"]
        H = best_hp["H"]
        N_lay = best_hp["layers"]
        D = best_hp["D"]
        F_dim = len(feat_cols)

        # Create windowed data.
        X_tr, y_tr, X_te, y_te, r_te = scale_and_window(train, test, feat_cols, L)
        print(f"  X_train: {X_tr.shape}, X_test: {X_te.shape}")

        # For final training, use last CV fold as early-stopping monitor.
        tscv = TimeSeriesSplit(n_splits=5)
        folds = list(tscv.split(np.arange(len(X_tr))))
        tr_idx, va_idx = folds[-1]

        model = LSTMClassifier(F_dim, H, N_lay, D, use_attention=use_attn).to(device)
        model, best_val = train_model(
            model, X_tr[tr_idx], y_tr[tr_idx], X_tr[va_idx], y_tr[va_idx])
        print(f"  Best val loss: {best_val:.4f}")

        # Retrain on full train set with the number of epochs from early stopping.
        # (Simplified: just use the model from the last fold — acceptable for thesis.)

        y_pred, attn_weights = predict_model(model, X_te)
        y_pred_labels = [LABEL_NAMES[i] for i in y_pred]

        results = bootstrap_ci(y_te, y_pred, y_pred_labels, r_te)
        results["hyperparameters"] = best_hp
        all_results[name] = results
        predictions[name] = {"y_true": y_te, "y_pred": y_pred}

        print(f"  DA:     {results['DA']['point']:.4f} "
              f"[{results['DA']['ci_low']:.4f}, {results['DA']['ci_high']:.4f}]")
        print(f"  F1:     {results['macro_F1']['point']:.4f} "
              f"[{results['macro_F1']['ci_low']:.4f}, {results['macro_F1']['ci_high']:.4f}]")
        print(f"  Sharpe: {results['Sharpe']['point']:.4f} "
              f"[{results['Sharpe']['ci_low']:.4f}, {results['Sharpe']['ci_high']:.4f}]")

        # Save attention heatmap for the attention + full model.
        if use_attn and feat_label == "full" and attn_weights is not None:
            fig, ax = plt.subplots(figsize=(12, 4))
            # Show attention for last 20 test windows.
            sample = attn_weights[-20:]
            sns.heatmap(sample, ax=ax, cmap="YlOrRd", xticklabels=range(1, L+1))
            ax.set_xlabel("Timestep in window")
            ax.set_ylabel("Test sample (last 20)")
            ax.set_title("Bahdanau Attention Weights — LSTM+Attention (full features)")
            plt.tight_layout()
            fig.savefig(f"{FIG_DIR}/attention_weights_sample.png", dpi=150)
            plt.close(fig)
            print("  Saved attention heatmap.")


Experiment: LSTM_full
  X_train: (737, 10, 230), X_test: (240, 10, 230)
  Best val loss: 1.1003
  DA:     0.3750 [0.3208, 0.4375]
  F1:     0.3208 [0.2700, 0.3759]
  Sharpe: 0.6946 [-1.1950, 2.7274]

Experiment: Attn_full
  X_train: (737, 10, 230), X_test: (240, 10, 230)
  Best val loss: 1.1006
  DA:     0.3167 [0.2583, 0.3792]
  F1:     0.2038 [0.1667, 0.2418]
  Sharpe: -1.1177 [-3.0665, 0.9198]
  Saved attention heatmap.

Experiment: LSTM_price
  X_train: (737, 10, 5), X_test: (240, 10, 5)
  Best val loss: 1.0988
  DA:     0.4625 [0.4042, 0.5251]
  F1:     0.2899 [0.2467, 0.3332]
  Sharpe: 3.4935 [1.6423, 5.3751]

Experiment: Attn_price
  X_train: (737, 10, 5), X_test: (240, 10, 5)
  Best val loss: 1.0966
  DA:     0.4667 [0.4042, 0.5333]
  F1:     0.3098 [0.2640, 0.3545]
  Sharpe: 3.6523 [1.6521, 5.6060]


## 8. McNemar tests

In [10]:
primary = "Attn_full"
baselines = ["LSTM_full", "LSTM_price", "Attn_price"]

print(f"Primary model: {primary}")
print()
for bl in baselines:
    if bl in predictions and primary in predictions:
        # Align lengths (may differ by a few due to windowing differences — use min).
        n = min(len(predictions[primary]["y_true"]), len(predictions[bl]["y_true"]))
        result = mcnemar_test(
            predictions[primary]["y_true"][:n],
            predictions[primary]["y_pred"][:n],
            predictions[bl]["y_pred"][:n],
        )
        print(f"  {primary} vs {bl}: "
              f"χ²={result['statistic']:.3f}, p={result['p_value']:.4f}")
        all_results[primary].setdefault("mcnemar", {})[bl] = result

Primary model: Attn_full

  Attn_full vs LSTM_full: χ²=2.112, p=0.1461
  Attn_full vs LSTM_price: χ²=7.092, p=0.0077
  Attn_full vs Attn_price: χ²=7.853, p=0.0051


## 9. Confusion matrices

In [11]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for ax, name in zip(axes.flat, ["LSTM_full", "LSTM_price", "Attn_full", "Attn_price"]):
    if name in predictions:
        cm = confusion_matrix(predictions[name]["y_true"], predictions[name]["y_pred"])
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                    xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES)
        ax.set_title(name)
        ax.set_ylabel("Actual")
        ax.set_xlabel("Predicted")
plt.suptitle("Confusion Matrices — LSTM Experiments", fontsize=14)
plt.tight_layout()
fig.savefig(f"{FIG_DIR}/confusion_matrices.png", dpi=150)
plt.close(fig)
print("Saved confusion matrices.")

Saved confusion matrices.


## 10. Save LSTM results

In [12]:
def _convert(obj):
    import numpy as np2
    if isinstance(obj, (np2.integer,)): return int(obj)
    if isinstance(obj, (np2.floating,)): return float(obj)
    if isinstance(obj, np2.ndarray): return obj.tolist()
    return obj

# Remove non-serialisable prediction arrays before saving.
save_results = {k: {kk: vv for kk, vv in v.items()}
                for k, v in all_results.items()}

clean = json.loads(json.dumps(save_results, default=_convert))
with open(RESULTS_PATH, "w") as f:
    json.dump(clean, f, indent=2)
print(f"Saved {RESULTS_PATH}")

Saved /kaggle/working/eval_results_lstm.json


## 11. Consolidated 2×4 ablation table

Loads tree baseline results from Notebook 10 and combines with the LSTM
results from this notebook to produce the final thesis table.

In [13]:
# Load baseline results.
if os.path.exists(BASELINE_RESULTS):
    with open(BASELINE_RESULTS) as f:
        baseline = json.load(f)
    print("Loaded baseline results from Notebook 10.")
else:
    print("WARNING: baseline results not found. Using placeholder.")
    baseline = {}

# Merge.
combined = {**baseline, **all_results}

# Print the 2x4 table.
models = ["RF", "XGB", "LSTM", "Attn"]
feats  = ["full", "price"]

def _fmt(name, metric):
    if name not in combined:
        return "---"
    r = combined[name]
    if metric not in r:
        return "---"
    v = r[metric]
    if isinstance(v, dict):
        return f"{v['point']:.3f} [{v['ci_low']:.3f},{v['ci_high']:.3f}]"
    return f"{v:.3f}"

print()
print("=" * 100)
print("ABLATION TABLE — Directional Accuracy")
print("=" * 100)
header = f"{'Features':<12s}" + "".join(f"{m:>24s}" for m in models)
print(header)
print("-" * 100)
for feat in feats:
    row = f"{feat:<12s}"
    for m in models:
        name = f"{m}_{feat}"
        row += f"{_fmt(name, 'DA'):>24s}"
    print(row)

print()
print("=" * 100)
print("ABLATION TABLE — Sharpe Ratio")
print("=" * 100)
print(header)
print("-" * 100)
for feat in feats:
    row = f"{feat:<12s}"
    for m in models:
        name = f"{m}_{feat}"
        row += f"{_fmt(name, 'Sharpe'):>24s}"
    print(row)

if "Buy_and_Hold" in combined:
    bnh = combined["Buy_and_Hold"]["Sharpe"]
    bnh_val = bnh["point"] if isinstance(bnh, dict) else bnh
    print(f"\nBuy-and-Hold Sharpe: {bnh_val:.4f}")

print()
print("=" * 100)
print("ABLATION TABLE — Macro-F1")
print("=" * 100)
print(header)
print("-" * 100)
for feat in feats:
    row = f"{feat:<12s}"
    for m in models:
        name = f"{m}_{feat}"
        row += f"{_fmt(name, 'macro_F1'):>24s}"
    print(row)


ABLATION TABLE — Directional Accuracy
Features                          RF                     XGB                    LSTM                    Attn
----------------------------------------------------------------------------------------------------
full                             ---                     ---     0.375 [0.321,0.438]     0.317 [0.258,0.379]
price                            ---                     ---     0.463 [0.404,0.525]     0.467 [0.404,0.533]

ABLATION TABLE — Sharpe Ratio
Features                          RF                     XGB                    LSTM                    Attn
----------------------------------------------------------------------------------------------------
full                             ---                     ---    0.695 [-1.195,2.727]   -1.118 [-3.066,0.920]
price                            ---                     ---     3.494 [1.642,5.375]     3.652 [1.652,5.606]

ABLATION TABLE — Macro-F1
Features                          RF           

---

**Done.** The 2×4 ablation table is the centrepiece of the thesis results chapter.

**How to read it:**

- **Columns** (left → right): increasing model complexity. If Attn beats LSTM
  consistently → Novelty 2 (attention) adds value.
- **Rows** (full vs. price): if full beats price consistently → Novelty 1
  (FinBERT sentiment features) adds value.
- **Buy-and-Hold Sharpe**: the bar any model must beat to have practical value.

All figures are saved in `/kaggle/working/figures/`. The thesis can include them
directly.

In [14]:
# ── Save predictions into features_targets for the dashboard ──────────────
import pandas as pd

# Load the features_targets parquet (from Step 4 input)
ft = pd.read_parquet("/kaggle/input/datasets/leev75/pfe-step41/features_targets.parquet")

# The test set in this notebook is windowed (loses first seq_len-1 rows).
# We align by taking the LAST len(y_pred) rows of the test split.
test_mask  = ft["split"] == "test"
test_dates = ft[test_mask]["trading_day"].values  # all test dates
n_test_windowed = len(predictions["LSTM_full"]["y_pred"])  # length after windowing
# Alignment: windowed predictions match the LAST n_test_windowed rows of test
aligned_idx = ft[test_mask].iloc[-n_test_windowed:].index

for exp_name, col_name in [
    ("LSTM_full",  "pred_lstm_full"),
    ("LSTM_price", "pred_lstm_price"),
    ("Attn_full",  "pred_attn_full"),
    ("Attn_price", "pred_attn_price"),
]:
    pred_ints   = predictions[exp_name]["y_pred"]          # integer array
    pred_labels = [LABEL_NAMES[i] for i in pred_ints]     # "BUY"/"HOLD"/"SELL"
    ft.loc[aligned_idx, col_name] = pred_labels

# Save to working directory
ft.to_parquet("/kaggle/working/features_targets.parquet", index=False)
print("✅ LSTM/Attn predictions saved.")
print(ft.loc[aligned_idx, ["trading_day","pred_lstm_price","pred_attn_price"]].tail(5))

✅ LSTM/Attn predictions saved.
    trading_day pred_lstm_price pred_attn_price
990  2023-12-21             BUY             BUY
991  2023-12-22             BUY             BUY
992  2023-12-26             BUY             BUY
993  2023-12-27             BUY             BUY
994  2023-12-28             BUY             BUY
